In [0]:
%run ./config

In [0]:
population = spark.table(f"{CATALOG}.{SCHEMA}.slv_population")
locations = spark.table(f"{CATALOG}.{SCHEMA}.slv_locations")

In [0]:
country_reference = (population.select("country_name", "country_iso3").dropDuplicates())

In [0]:
country_overrides =  spark.createDataFrame(
    [("United States", "USA"),
     ("South Korea", "KOR"),
     ("Korea, Republic of", "KOR"),
     ("Russia", "RUS"),
     ("Iran", "IRN"),
     ("Vietnam", "VNM"),
     ("Czechia", "CZE"),
     ("Taiwan", "TWN")],
    ["country_name", "country_iso3"])

In [0]:
country_mapping = (
    country_reference.unionByName(country_overrides)
                     .withColumn("country_normalized",F.lower(F.trim("country_name"))).dropDuplicates(["country_normalized"]))

In [0]:
locations_mapped = (
    locations.alias("location").join(country_mapping.alias("mapping"),F.lower(F.trim(F.col("location.country_name"))) == F.col("mapping.country_normalized"),"left")
    .select("location.*",F.col("mapping.country_iso3").alias("country_iso3")))

In [0]:
locations_mapped.write.format("delta")\
                      .mode("overwrite")\
                      .option("overwriteSchema", "true")\
                      .saveAsTable(f"{CATALOG}.{SCHEMA}.slv_locations_iso3")

In [0]:
%sql
SELECT * FROM mvp_eng_dados.mvp_cancer.slv_locations_iso3